# Dispatch Visualization Draft

This notebook rebuilds the optimizer decision variables from the saved daily inputs and renders an interactive dispatch dashboard.

In [1]:
from datetime import date
from pathlib import Path
import os
import sys

cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent, cwd.parent.parent]
PROJECT_ROOT = next(
    candidate for candidate in candidates if (candidate / "energy_planner" / "src").exists()
)
SRC_ROOT = PROJECT_ROOT / "energy_planner" / "src"

for candidate in (PROJECT_ROOT, SRC_ROOT):
    candidate_str = str(candidate)
    if candidate_str not in sys.path:
        sys.path.insert(0, candidate_str)

import pandas as pd
from IPython.display import Markdown, display

from ingestion.load_predicted_inputs import load_predicted_inputs
from main import _run_optimizer
from reporting.optimization_summary import (
    build_optimization_summary_payload,
    try_generate_llm_summary,
)
from state.load_state import load_current_state
from visualization import (
    build_visualization_frame,
    create_dispatch_dashboard,
    save_dashboard_html,
    summarize_dispatch,
)


In [ ]:
run_date = date(2026, 3, 11)
llm_base_url = "https://openrouter.ai/api/v1"
llm_model = "openrouter/free"
llm_api_key = "TO SET"


if not llm_api_key or llm_api_key == "PUT_YOUR_OPENROUTER_API_KEY_HERE":
    raise RuntimeError(
        "Missing API key. Replace the placeholder with your own OpenRouter key before running this notebook."
    )


predicted_inputs = load_predicted_inputs(run_date=run_date)
state = load_current_state(run_date=run_date)
plan_df = _run_optimizer(predicted_inputs, state)

if plan_df is None:
    raise RuntimeError("Optimizer plan could not be created. Check the CPLEX installation.")

viz_df = build_visualization_frame(
    predicted_inputs,
    plan_df,
    initial_battery_kwh=state["E_bat_0"],
    battery_capacity_kwh=state["E_max"],
)

summary = summarize_dispatch(viz_df)
summary_payload = build_optimization_summary_payload(
    predicted_inputs,
    plan_df,
    initial_battery_kwh=state["E_bat_0"],
    battery_capacity_kwh=state["E_max"],
)
summary_text, summary_source = try_generate_llm_summary(
    summary_payload,
    model=llm_model,
    api_key=llm_api_key,
    base_url=llm_base_url,
)
summary_markdown = "## Resume naturel du dispatch\n\n" \
    + f"Source: `{summary_source}` | Modele: `{llm_model}`\n\n" \
    + summary_text
summary


Statut de la solution : 101 - integer optimal solution
Valeur de la fonction objectif : -30.9013722

Heure | Pin  | Pgo  | PV   | Pch | Pdis | Ebat | S
    0 | 5.63 | 0.00 | 0.10 | 4.00 | 0.00 | 10.00 | 1
    1 | 3.98 | 0.00 | 0.01 | 2.17 | 0.00 | 12.17 | 1
    2 | 0.00 | 0.00 | 0.04 | 0.00 | 1.72 | 10.45 | 1
    3 | 0.00 | 0.00 | 0.10 | 0.00 | 1.68 | 8.77 | 1
    4 | 0.00 | 0.00 | 0.00 | 0.00 | 1.78 | 7.00 | 1
    5 | 0.00 | 0.00 | 0.00 | 0.00 | 1.65 | 5.34 | 1
    6 | 0.00 | 0.00 | 0.00 | 0.00 | 2.27 | 3.08 | 1
    7 | 0.00 | 0.00 | 0.64 | 0.00 | 1.82 | 1.25 | 1
    8 | 0.00 | 0.00 | 1.30 | 0.00 | 1.25 | 0.00 | 1
    9 | 0.00 | 0.00 | 2.44 | 0.07 | 0.00 | 0.07 | 1
   10 | 0.00 | 0.00 | 3.15 | 1.09 | 0.00 | 1.16 | 1
   11 | 0.00 | 0.00 | 4.47 | 2.73 | 0.00 | 3.89 | 1
   12 | 0.00 | 0.00 | 4.86 | 3.38 | 0.00 | 7.26 | 1
   13 | 0.00 | 0.00 | 5.35 | 3.93 | 0.00 | 11.20 | 1
   14 | 0.00 | 1.36 | 5.13 | 2.30 | 0.00 | 13.50 | 1
   15 | 0.00 | 2.71 | 4.41 | 0.00 | 0.00 | 13.50 | 1
   16 | 0.

{'total_served_demand_kWh': 48.636,
 'total_pv_kWh': 39.804,
 'grid_import_kWh': 9.614,
 'grid_export_kWh': 6.782,
 'battery_charge_kWh': 19.674,
 'battery_discharge_kWh': 25.674,
 'peak_battery_kWh': 13.5,
 'regime_hours': {'Battery support': 12,
  'Grid charge': 2,
  'PV export': 5,
  'Solar charge': 5}}

In [3]:
viz_df[
    [
        "hour",
        "Pfix_pred_kW",
        "Pflex_pred_kW",
        "total_demand_kW",
        "PV",
        "Pin",
        "Pgo",
        "Pch",
        "Pdis",
        "Ebat",
        "regime",
    ]
]


,hour,Pfix_pred_kW,Pflex_pred_kW,total_demand_kW,PV,Pin,Pgo,Pch,Pdis,Ebat,regime
0,0,1.485,0.251,1.736,0.102,5.634,0.000,4.000,0.000,10.000,Grid charge
1,1,1.499,0.317,1.816,0.010,3.980,0.000,2.174,0.000,12.174,Grid charge
2,2,1.455,0.308,1.763,0.043,0.000,0.000,0.000,1.720,10.454,Battery support
3,3,1.423,0.353,1.776,0.095,0.000,0.000,0.000,1.681,8.773,Battery support
4,4,1.429,0.348,1.777,0.000,0.000,0.000,0.000,1.777,6.996,Battery support
5,5,1.196,0.455,1.651,0.000,0.000,0.000,0.000,1.651,5.345,Battery support
6,6,1.128,1.142,2.270,0.000,0.000,0.000,0.000,2.270,3.075,Battery support
7,7,1.088,1.371,2.459,0.635,0.000,0.000,0.000,1.824,1.251,Battery support
8,8,1.040,1.516,2.556,1.305,0.000,0.000,0.000,1.251,0.000,Battery support
9,9,1.016,1.355,2.371,2.444,0.000,0.000,0.073,0.000,0.073,Solar charge


In [4]:
fig = create_dispatch_dashboard(
    viz_df,
    # title=f"Smart Building Dispatch Dashboard | {run_date.isoformat()}",
)
fig.show()


In [5]:
output_path = PROJECT_ROOT / "energy_planner" / "data" / "processed" / f"dispatch_dashboard_{run_date.isoformat()}.html"
saved = save_dashboard_html(fig, output_path)
saved


WindowsPath('C:/Users/barra/Desktop/Cours 3A CentraleSupelec/Projet Dassault/Projet-Dassault-Smart-Building/energy_planner/data/processed/dispatch_dashboard_2026-03-11.html')

In [6]:
display(Markdown(summary_markdown))


## Resume naturel du dispatch

Source: `llm` | Modele: `openrouter/free`

Voici une explication détaillée de la gestion des sources d'énergie pour ce smart building :

La demande totale d'énergie était de 48.636 kWh. Le système a couvert cette demande de manière optimisée en utilisant plusieurs sources d'énergie.

L'énergie photovoltaïque (PV) a produit 39.804 kWh. Parmi cette production, 33.626 kWh ont été autoconsommés directement, ce qui représente un taux d'autoconsommation PV de 84.5 %. Le reste de la production PV a été exporté vers le réseau.

La batterie a joué un rôle crucial dans la gestion de l'énergie. Elle a été chargée avec 19.674 kWh d'énergie et déchargée avec 25.674 kWh. La batterie a commencé la période avec 6.0 kWh et s'est terminée à 0.0 kWh, atteignant un maximum de 13.5 kWh.

Pour couvrir la demande totale, 9.614 kWh ont dû être achetés au réseau principal (P_in). Le coût de cette énergie importée s'est élevé à 1.631 EUR. Il est important de noter que ce coût ne concerne que l'énergie importée, et non toute la demande servie.

Le système a également généré des revenus en revendant de l'énergie au réseau. Un total de 6.782 kWh a été vendu (P_go), générant un revenu de 0.874 EUR.

Le coût net de l'énergie pour cette période s'élève à 0.757 EUR, calculé comme la différence entre le coût d'achat et le revenu de vente.

Le système a également géré 20.611 kWh de flexibilité énergétique, avec un taux de service de 100.0 %, ce qui signifie que toute la flexibilité demandée a été satisfaite.

En termes de pics de consommation et de production :
- Le pic d'achat au réseau s'est produit à l'heure 0 avec une valeur de 5.634 kW.
- Le pic de vente au réseau s'est produit à l'heure 15 avec une valeur de 2.713 kW.
- Le pic de charge de la batterie s'est produit à l'heure 0 avec une valeur de 4.0 kW.
- Le pic de décharge de la batterie s'est produit à l'heure 19 avec une valeur de 3.272 kW.

La répartition des régimes horaires montre que la batterie a fourni un support pendant 12 heures, le réseau a chargé la batterie pendant 2 heures, il y a eu exportation de l'énergie PV pendant 5 heures, et la batterie a été chargée par l'énergie solaire pendant 5 heures.

En conclusion, ce système de gestion d'énergie autonome a efficacement utilisé les sources d'énergie disponibles pour répondre à la demande, minimisant ainsi les coûts et maximisant l'utilisation de l'énergie renouvelable.